[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-06-sentiment-naive-bayes.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 6 — Sentiment analysis with Naïve Bayes

**MSIS · AME 5053 · Week 6 · 3 hours**

Session 16 classified one film review, by hand, on five training sentences. This lab
builds the same classifier in code, checks it against the numbers from the board, and
then points it at **2,000 real movie reviews**.

Then it does the part that lecture had no time for: **measuring whether the classifier
is any good**, and finding out how much you can trust the measurement.

**By the end of this lab you will be able to:**

1. Reproduce session 16's worked example exactly, in code
2. Watch the unsmoothed classifier fail, and fix it with add-one
3. Train Naïve Bayes on a real corpus, in log space
4. Get the same answer from scikit-learn, and explain any difference
5. Read a precision/recall report and a confusion matrix
6. Show that a single accuracy number is not enough to compare two models

Run every cell, in order.

---

## Part 0 — Setup

NLTK is already installed on Colab, but its **corpora are not** — those download separately,
and they vanish when Colab recycles the machine. So this cell runs every session.

In [ ]:
!pip install -q nltk scikit-learn

import nltk

# No quiet=True: if the download fails we want to see it, not carry on with no data.
ok = nltk.download("movie_reviews")
print("movie_reviews downloaded:", ok)

from nltk.corpus import movie_reviews
print("categories:", movie_reviews.categories())
print("Done.")

> **Save your own copy now:** File → Save a copy in Drive.

If `movie_reviews downloaded: False` above, re-run the cell — it is almost always a network
hiccup. Parts 1 and 2 will work without it, but nothing from Part 3 onwards will.

---

## Part 1 — Session 16, in code

Before we touch 2,000 reviews, we rebuild the tiny example from the lecture. It is small
enough that you can check every number by hand, and the cells `assert` the results — so if
your code disagrees with the board, you find out immediately.

Here is the training data. Five documents, three negative and two positive.

In [ ]:
neg_docs = [
    "just plain boring",
    "entirely predictable and lacks energy",
    "no surprises and very few laughs",
]

pos_docs = [
    "very powerful",
    "the most fun film of the summer",
]

for d in neg_docs:
    print("-", d)
for d in pos_docs:
    print("+", d)

### Counting

Naïve Bayes training is nothing but counting. We need, for each class: how many times each
word appears, and how many word tokens there are in total. We also need the **vocabulary**
— and this is the first place people go wrong.

The vocabulary is the set of words across **both classes together**, not one set per class.
Session 16 flagged this as a trap; here the code makes it concrete.

In [ ]:
from collections import Counter

neg_tokens = [w for d in neg_docs for w in d.split()]
pos_tokens = [w for d in pos_docs for w in d.split()]

neg_counts = Counter(neg_tokens)
pos_counts = Counter(pos_tokens)

# The vocabulary is the union across BOTH classes.
V = sorted(set(neg_tokens) | set(pos_tokens))

print("negative tokens:", len(neg_tokens))
print("positive tokens:", len(pos_tokens))
print("|V| =", len(V))

assert len(neg_tokens) == 14
assert len(pos_tokens) == 9   # 'the' appears twice — 9 tokens, not 8
assert len(V) == 20
print("\nMatches the lecture.")

### Add-one smoothing

Session 13 condemned add-one smoothing for language models, where the vocabulary is 50,000
words. Here |V| is 20, and add-one is exactly the right tool. The method did not change —
the vocabulary size did.

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{(total tokens in } c) + |V|}$$

> **A small display detail.** Python's `Fraction` reduces automatically, so where the lecture
> wrote `2/34` you will see `1/17`. Same number, printed in lowest terms.

In [ ]:
from fractions import Fraction

def P(word, counts, total):
    """P(word | class), with add-one smoothing."""
    return Fraction(counts[word] + 1, total + len(V))

print(f"{'word':14}{'P(w|-)':>10}{'P(w|+)':>10}")
for w in ["predictable", "no", "fun"]:
    print(f"{w:14}{str(P(w, neg_counts, 14)):>10}{str(P(w, pos_counts, 9)):>10}")

# 2/34 and 2/29 from the board, reduced.
assert P("predictable", neg_counts, 14) == Fraction(2, 34)
assert P("fun", pos_counts, 9) == Fraction(2, 29)

### Scoring the test document

The test document is **"predictable with no fun"**.

`with` is not in the vocabulary, so we drop it. Remember why: a classifier only ever
*compares* classes, and an unknown word would contribute the same factor to both, so it
cannot change which class wins. Dropping it is not laziness — it is the right thing to do.

The priors are 3/5 negative and 2/5 positive, straight from the training data.

In [ ]:
prior_neg = Fraction(3, 5)
prior_pos = Fraction(2, 5)

test_doc = "predictable with no fun".split()
print("dropped (not in V):", [w for w in test_doc if w not in V])

def score(counts, total, prior):
    s = prior
    for w in test_doc:
        if w in V:
            s *= P(w, counts, total)
    return s

s_neg = score(neg_counts, 14, prior_neg)
s_pos = score(pos_counts, 9, prior_pos)

print("\nP(-) * likelihoods =", s_neg, "=", float(s_neg))
print("P(+) * likelihoods =", s_pos, "=", float(s_pos))
print("\nwinner:", "negative" if s_neg > s_pos else "positive")

assert s_neg == Fraction(3, 49130)
assert s_pos == Fraction(4, 121945)
assert s_neg > s_pos

### Your turn

The word `with` was dropped. Prove to yourself that dropping it was safe: score the test
document again, but this time **pretend `with` is in the vocabulary** with a count of 0 in
both classes, and check whether the winner changes.

In [ ]:
# YOUR CODE HERE
# Multiply each class score by its add-one probability for an unseen 'with',
# then compare the two scores again.

---

## Part 2 — What happens without smoothing

Session 16's best moment. Take the `+ 1` out and score the same document again.

In [ ]:
def P_unsmoothed(word, counts, total):
    return Fraction(counts[word], total)

def score_unsmoothed(counts, total, prior):
    s = prior
    for w in test_doc:
        if w in V:
            s *= P_unsmoothed(w, counts, total)
    return s

print("negative:", score_unsmoothed(neg_counts, 14, prior_neg))
print("positive:", score_unsmoothed(pos_counts, 9, prior_pos))

print("\nwhich word killed each class?")
print("  negative has never seen:", [w for w in test_doc if w in V and neg_counts[w] == 0])
print("  positive has never seen:", [w for w in test_doc if w in V and pos_counts[w] == 0])

**Both classes are zero.**

This is worse than a wrong answer. The classifier does not prefer negative over positive —
it assigns zero probability to *both*, so it has no answer at all. One unseen word in one
class is enough: `fun` never appears in a negative review, `predictable` and `no` never
appear in a positive one, and each missing word drives its whole class to zero.

Add-one fixes it for the reason session 13 gave: it makes sure nothing is ever exactly
impossible.

---

## Part 3 — Real data

2,000 movie reviews, 1,000 positive and 1,000 negative, collected by Pang and Lee. This is
the dataset the sentiment section of the textbook is built around.

In [ ]:
texts = [" ".join(movie_reviews.words(f))
         for c in movie_reviews.categories()
         for f in movie_reviews.fileids(c)]

labels = [c for c in movie_reviews.categories()
            for f in movie_reviews.fileids(c)]

print("documents:", len(texts))
print("positive: ", labels.count("pos"))
print("negative: ", labels.count("neg"))
print("total word tokens:", len(movie_reviews.words()))
print("word types:        ", len(set(movie_reviews.words())))

In [ ]:
# Look at one. Always look at your data before you model it.
print("label:", labels[0])
print(texts[0][:700], "...")

### Splitting

We train on 1,500 reviews and test on 500 the model has never seen. Two details that look
like boilerplate and are not:

- `stratify=labels` keeps the 50/50 balance in both halves. Without it the split could hand
  you a lopsided test set by chance.
- `random_state=42` makes the split reproducible. **Part 6 is about how much this one number
  matters** — more than you would guess.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels)

print("train:", len(X_train), " test:", len(X_test))
print("train balance:", y_train.count("pos"), "pos /", y_train.count("neg"), "neg")
print("test balance: ", y_test.count("pos"), "pos /", y_test.count("neg"), "neg")

---

## Part 4 — Naïve Bayes from scratch, at scale

The same algorithm as Part 1, pointed at 1,500 documents. One thing must change.

A review is around 700 words. Scoring it multiplies ~700 probabilities, each well below 1.
The result underflows to 0.0 in floating point long before you reach the end, and then every
class ties at zero and the classifier is useless.

So we work in **log space**, exactly as sessions 12, 14 and 16 promised. Multiplication
becomes addition, and the comparison still works because log is increasing — whichever class
has the larger log score has the larger probability.

Here is the training function. Training is just counting, so it is given to you.

In [ ]:
import math

def train_nb(docs, doc_labels):
    """Count everything Naive Bayes needs. Returns a model tuple."""
    classes = sorted(set(doc_labels))
    counts = {c: Counter() for c in classes}
    vocab = set()

    for d, c in zip(docs, doc_labels):
        words = d.split()
        counts[c].update(words)
        vocab.update(words)

    logprior = {c: math.log(doc_labels.count(c) / len(doc_labels)) for c in classes}
    totals = {c: sum(counts[c].values()) for c in classes}
    return classes, logprior, counts, totals, vocab

model = train_nb(X_train, y_train)
classes, logprior, counts, totals, vocab = model

print("classes:", classes)
print("vocabulary:", len(vocab))
print("tokens per class:", {c: totals[c] for c in classes})

### Your turn — write the prediction step

This is the part that matters. For each class, start from its log prior and **add** the log
probability of every word in the document. Skip words that are not in the vocabulary. Return
the class with the highest total.

The smoothed probability of a word, as in Part 1:

```
(counts[c][w] + 1) / (totals[c] + len(vocab))
```

In [ ]:
def predict_nb(doc, model):
    classes, logprior, counts, totals, vocab = model
    # YOUR CODE HERE
    # For each class: start at logprior[c], add math.log(...) for each known word.
    # Return the class with the highest score.
    pass

In [ ]:
predictions = [predict_nb(d, model) for d in X_test]

correct = sum(p == t for p, t in zip(predictions, y_test))
acc_scratch = correct / len(y_test)

print(f"from scratch: {correct}/{len(y_test)} correct = {acc_scratch:.3f}")

# 0.816 on the machine this lab was written on. Unlike the Part 1 fractions,
# this number depends on library versions, so we only check it is in the right
# neighbourhood — a working model lands near 0.8, a broken one near 0.5.
assert 0.75 < acc_scratch < 0.90, "that is not a working classifier — check predict_nb"

About 82%, from roughly twenty lines of counting. Random guessing on a balanced set would
get 50%.

Hold on to that number. We are about to get a different one.

---

## Part 5 — The same thing in scikit-learn

Two objects. `CountVectorizer` turns documents into a matrix of word counts — session 17's
term-document matrix, transposed. `MultinomialNB` is Naïve Bayes.

Note `alpha=1.0`: that is add-one smoothing, and it is the default. You have already
implemented it.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

print("matrix shape:", X_train_vec.shape, "(documents x words)")

nb = MultinomialNB(alpha=1.0).fit(X_train_vec, y_train)
acc_sklearn = nb.score(X_test_vec, y_test)

print(f"\nfrom scratch: {acc_scratch:.3f}")
print(f"scikit-learn: {acc_sklearn:.3f}")

### They disagree

Same algorithm. Same smoothing. Same training data. Same test set. Different answers.

**Stop and think before running the next cell.** Only one thing can differ. What is it?

In [ ]:
print("our vocabulary:    ", len(vocab))
print("sklearn vocabulary:", len(vec.vocabulary_))

only_ours = sorted(vocab - set(vec.vocabulary_))
print("\nin ours but not sklearn's:", len(only_ours), "items")
print("a sample:", ", ".join(repr(w) for w in only_ours[:10]))

It is **tokenization** — the step before any of the modelling.

We used `.split()`, which cuts on whitespace and keeps everything else. `CountVectorizer`
applies a regular expression that keeps only runs of two or more word characters, so it
drops punctuation and single characters.

Watch what that does to a sentence:

In [ ]:
analyzer = vec.build_analyzer()

sentence = "the movie was n't good ."
print("our .split() :", sentence.split())
print("sklearn      :", analyzer(sentence))

**Look at what vanished.** Not just the full stop — `n't` is gone too. The pattern needs
**two word characters in a row**, and the apostrophe splits `n't` into a lone `n` and a lone
`t`, so neither half survives. scikit-learn's default tokenizer throws away the negation.

Remember that for Part 7.

### Closing the gap

If tokenization is the only difference, then giving our model sklearn's tokenizer should
make the two agree exactly. `vec.build_analyzer()` hands us the exact function it uses.

In [ ]:
def train_nb_tok(docs, doc_labels, tokenize):
    classes = sorted(set(doc_labels))
    counts = {c: Counter() for c in classes}
    vocab = set()
    for d, c in zip(docs, doc_labels):
        words = tokenize(d)
        counts[c].update(words)
        vocab.update(words)
    logprior = {c: math.log(doc_labels.count(c) / len(doc_labels)) for c in classes}
    totals = {c: sum(counts[c].values()) for c in classes}
    return classes, logprior, counts, totals, vocab

def predict_nb_tok(doc, model, tokenize):
    classes, logprior, counts, totals, vocab = model
    best_class, best_score = None, None
    for c in classes:
        s = logprior[c]
        for w in tokenize(doc):
            if w in vocab:
                s += math.log((counts[c][w] + 1) / (totals[c] + len(vocab)))
        if best_score is None or s > best_score:
            best_class, best_score = c, s
    return best_class

model2 = train_nb_tok(X_train, y_train, analyzer)
preds2 = [predict_nb_tok(d, model2, analyzer) for d in X_test]
acc_matched = sum(p == t for p, t in zip(preds2, y_test)) / len(y_test)

print(f"ours, sklearn's tokenizer: {acc_matched:.4f}   vocab {len(model2[4])}")
print(f"scikit-learn:              {acc_sklearn:.4f}   vocab {len(vec.vocabulary_)}")

assert len(model2[4]) == len(vec.vocabulary_)
assert round(acc_matched, 4) == round(acc_sklearn, 4)
print("\nIdentical. The algorithms never disagreed — the tokenizers did.")

That is the lesson of Part 5, and it is worth more than the accuracy number:

> **A difference in results is not always a difference in the model.** Before you conclude
> that one method beats another, check that they were given the same input. Here the
> "better" library was doing nothing cleverer than dropping punctuation.

---

## Part 6 — Is 82% any good?

Accuracy is one number, and it hides things. A classifier can be right 82% of the time while
being much better at one class than the other.

Two tools:

- **Confusion matrix** — how many of each true class went to each predicted class.
- **Precision and recall.** For the positive class: *precision* is how many of the reviews
  you called positive really were; *recall* is how many of the truly positive reviews you
  found. **F1** is their harmonic mean, a single number when you need one.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = nb.predict(X_test_vec)

print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred, labels=["neg", "pos"])
print("confusion matrix (rows = true, columns = predicted)")
print("            pred neg   pred pos")
print(f"true neg   {cm[0][0]:>8}   {cm[0][1]:>8}")
print(f"true pos   {cm[1][0]:>8}   {cm[1][1]:>8}")

Every number in that report is the same: 0.824 precision, 0.824 recall, 0.824 F1, 0.824
accuracy.

That is not a coincidence, and it is not a bug. Look at the confusion matrix — it is
**symmetric**. The model made 44 mistakes in each direction, so it is wrong about positives
exactly as often as about negatives. When errors are balanced like this, accuracy really does
tell you everything.

That is unusual. In a moment we will look at a model where it is not true.

### Your turn

Compute precision and recall for the **positive** class straight from the confusion matrix,
without using scikit-learn, and check you get the same numbers.

With `cm` laid out as above: `cm[1][1]` is true positives, `cm[0][1]` is negatives wrongly
called positive, and `cm[1][0]` is positives wrongly called negative.

In [ ]:
# YOUR CODE HERE
# precision = true positives / everything you called positive
# recall    = true positives / everything that really was positive

### A second model, and a warning

The textbook says that for sentiment, whether a word appears matters more than how often —
so counts should be replaced by 0/1. `CountVectorizer(binary=True)` does that.

Let us test the claim.

In [ ]:
vec_bin = CountVectorizer(binary=True)
X_train_bin = vec_bin.fit_transform(X_train)
nb_bin = MultinomialNB().fit(X_train_bin, y_train)

X_test_bin = vec_bin.transform(X_test)
y_pred_bin = nb_bin.predict(X_test_bin)
acc_bin = nb_bin.score(X_test_bin, y_test)

print(classification_report(y_test, y_pred_bin, digits=3))
print(f"counts: {acc_sklearn:.3f}    binary: {acc_bin:.3f}")

**Now precision and recall come apart.** For `neg`: precision 0.812, recall 0.832. For `pos`:
precision 0.828, recall 0.808. The model leans towards calling things negative — it finds
83% of the negative reviews but only 81% of what it calls negative really is.

Accuracy alone would have hidden that, which is why you print the report.

And on this split, **binary scored *lower* than counts** — 0.820 against 0.824. So the
textbook is wrong?

No. We have not measured carefully enough to say anything at all yet.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

for binary in (False, True):
    pipe = make_pipeline(CountVectorizer(binary=binary), MultinomialNB())
    scores = cross_val_score(pipe, texts, labels, cv=10)
    name = "binary" if binary else "counts"
    print(f"{name:8} 10-fold mean {scores.mean():.4f}   std {scores.std():.4f}")

Ten different splits instead of one, and the ordering **reverses**: binary now comes out
ahead.

So which is it? Run the decisive test — the same 20 splits given to both models, and count
how often each wins.

*(This fits 40 models. It takes about half a minute.)*

In [ ]:
import numpy as np

diffs = []
for seed in range(20):
    a_tr, a_te, b_tr, b_te = train_test_split(
        texts, labels, test_size=0.25, random_state=seed, stratify=labels)
    got = {}
    for binary in (False, True):
        v = CountVectorizer(binary=binary)
        m = MultinomialNB().fit(v.fit_transform(a_tr), b_tr)
        got[binary] = m.score(v.transform(a_te), b_te)
    diffs.append(got[True] - got[False])

wins = sum(d > 0 for d in diffs)
ties = sum(d == 0 for d in diffs)

print(f"binary wins {wins}, ties {ties}, counts wins {20 - wins - ties}  (out of 20)")
print(f"mean difference {np.mean(diffs):+.4f}   std {np.std(diffs):.4f}")
print(f"range {min(diffs):+.3f} to {max(diffs):+.3f}")

### Read that carefully

Binary wins 11 of 20, ties 4, loses 5. The average gain is about **+0.007** — but the
standard deviation of the difference is **0.012**, nearly twice as large, and on individual
splits the difference runs from −0.02 to +0.03.

So the honest conclusion is not "binary is better" and not "counts is better":

> **The difference between these two models is smaller than the noise in a single split.**
> One experiment cannot tell them apart. Reporting either as the winner from one 75/25 split
> would be reporting a coin flip.

This is the whole reason cross-validation exists, and it is the most transferable thing in
this lab. Any accuracy you read in a paper, a blog post or your own notebook has error bars
around it, whether or not anybody printed them.

---

## Part 7 — What it gets wrong

Numbers tell you how often. They do not tell you *what kind* of mistake. For that you read
the failures.

In [ ]:
import random

wrong = [(true, pred, doc)
         for true, pred, doc in zip(y_test, y_pred, X_test)
         if true != pred]

print(f"{len(wrong)} of {len(y_test)} misclassified\n")

random.seed(0)
for true, pred, doc in random.sample(wrong, 3):
    print(f"--- true: {true}, predicted: {pred}")
    print(doc[:400], "...\n")

Read a few of these properly. The same patterns keep coming up:

- **Negation.** "not the disaster I expected" is a positive review made of negative words.
- **Comparison.** A review that spends three paragraphs describing a bad film in order to say
  this one is better.
- **Plot summary.** A positive review of a horror film is full of words like *murder*,
  *dead*, *terrifying*.
- **Sarcasm.** Words that mean their opposite, with nothing in the text to mark it.

Every one of these is the same failure, and it is not the classifier's fault. Session 17
showed it:

```
"the film was good, not bad"   ->   the·1 film·1 was·1 good·1 not·1 bad·1
"the film was bad, not good"   ->   the·1 film·1 was·1 good·1 not·1 bad·1
```

Opposite meanings, identical representations. The information was destroyed when we chose
bag of words — before Naïve Bayes ever saw the document. **No classifier can recover a
distinction that its representation threw away.**

And in Part 5 we saw it destroyed even earlier: sklearn's tokenizer deletes `n't` outright.

Getting word order back takes the rest of this course.

---

## What you built

1. Naïve Bayes from scratch, agreeing with the lecture to the exact fraction
2. The same classifier on 2,000 real reviews, in log space, at ~82%
3. The same result from scikit-learn — after you found the tokenizer difference
4. A precision/recall report and a confusion matrix, and the ability to compute them by hand
5. Evidence that a single accuracy number cannot separate two similar models

The last one is the one to keep. **Next lab** goes back to the vector space model: cosine
similarity and TF-IDF, from sessions 18 and 19.